# HMM library

everything the HMM needs lives here. notebook 04 imports it all with `%run 02_hmm.ipynb`

what's in here:
- from-scratch Gaussian HMM (forward-backward, Baum-Welch, Viterbi)
- from-scratch Gaussian Naive Bayes baseline
- pipeline helpers that train and evaluate one feature source

## todo
- [x] use top 5 features from pearsons, not PCA
- [x] add flags for warm init, dirichlet, n_restarts
- [ ] plot PCA scatter plots

### setup
- ~140 patients in OASIS-2, each with 2 to 5 clinic visits about a year apart
- every visit has clinical numbers (MMSE, brain volume, etc.) and a CDR label: 0 = healthy, 0.5 = very mild, 1 = mild dementia
- goal: predict CDR at each visit and see how patients move between stages over time

### why an HMM?
- there's a hidden state at each visit (the "real" disease stage) that we can't measure directly
- each hidden state produces noisy clinical numbers (the "emission")
- the next state only depends on the current one (Markov assumption, we just need to know where the patient is now)
- we use 3 hidden states, hoping they line up with CDR 0, 0.5, and 1

### what the model learns
- **start probs**: at the very first visit, how likely is each disease stage?
- **transition probs**: 3x3 table. if a patient is in stage X now, what's the chance of stage Y next visit?
- **emissions**: for each stage, what do the clinical features look like? (bell curve per feature)

### training (Baum-Welch)
1. start with a guess (we can "warm start" from per-CDR averages or go random)
2. forward-backward: for every visit, figure out how confident we are about which stage it is
3. update: use those confidences to recompute start probs, transitions, emissions
4. repeat until it stops improving
5. run from multiple starting points and keep the best (avoids bad local optima)

### decoding new patients (Viterbi)
- given a new sequence of visits, find the single most likely path through disease stages

### mapping states to CDR (Hungarian algorithm)
- hidden states are just numbered 0, 1, 2, they don't come with labels
- Hungarian algorithm finds the best one-to-one mapping between states and CDR levels
- not cheating: just figuring out which learned cluster is which real label

#### imports

In [145]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import linear_sum_assignment  # hungarian algorithm: finds best 1-to-1 matching of HMM states to CDR labels
from sklearn.decomposition import PCA
from sklearn.metrics import (balanced_accuracy_score, classification_report, confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (7, 4)
CDR_VALUES = np.array([0.0, 0.5, 1.0])

# CDR 0 -> 0 (healthy), CDR 0.5 -> 1 (very mild), CDR 1 -> 2 (mild dementia)
def cdr_to_class_idx(cdr_values):
    """cdr {0, 0.5, 1} to int class {0, 1, 2}"""
    return np.round(np.asarray(cdr_values, dtype=float) * 2).astype(int)

def log_sum_exp(values, axis=None):
    """numerically stable log(sum(exp(values))), used in the HMM forward-backward"""
    # subtract the max before exponentiating so we don't overflow on big values
    max_val = np.max(values, axis=axis, keepdims=True)
    return np.squeeze(max_val, axis=axis) + np.log(np.sum(np.exp(values - max_val), axis=axis))


#### the HMM class

just a container for the 4 things the model needs:
- `startprob_`: how likely each stage is at a patient's first visit (length 3)
- `transmat_`: 3x3 table, given current stage what's the chance of each stage next visit
- `means_`: average feature values for each stage (3 x n_features)
- `covars_`: how spread out each feature is per stage (3 x n_features, diagonal covariance)

we set these from outside before calling fit, then Baum-Welch refines them

In [146]:
class GaussianHMMScratch:
    """3-state Gaussian HMM with diagonal covariance, written from scratch."""
    def __init__(self, n_states=3, n_iter=600, tol=1e-5, random_state=42):
        self.n_states = n_states
        # max baum-welch rounds before we give up
        self.n_iter = n_iter
        # stop when the log-likelihood improves by less than this between rounds
        self.tol = tol
        self.random_state = random_state
        # set externally before fit (warm start)
        self.startprob_ = None
        self.transmat_ = None
        self.means_ = None
        self.covars_ = None


### emission probabilities

for each visit, how likely are the observed features under each disease stage?

we model each stage as a bell curve per feature (gaussian). we work in log space because the raw probabilities get astronomically tiny and would round to zero. log space turns multiplications into additions, keeps things stable.

In [147]:
def _log_emission_probs(self, X):
    """for each visit, compute how likely the observed features are under each disease stage.
    uses a gaussian (bell curve) for each feature independently."""
    n_states, n_features = self.means_.shape
    seq_len = X.shape[0]
    log_two_pi = np.log(2.0 * np.pi)
    log_probs = np.empty((seq_len, n_states))
    for state in range(n_states):
        # how far is each visit's features from this stage's average, scaled by how spread out the stage is
        diff = X - self.means_[state]
        variance = self.covars_[state]
        log_probs[:, state] = -0.5 * (n_features * log_two_pi + np.log(variance).sum() + ((diff * diff) / variance).sum(axis=1))
    return log_probs

GaussianHMMScratch._log_emission_probs = _log_emission_probs

### forward-backward

for one patient's visit sequence, figure out how confident we are about each visit's disease stage.

two passes:
- **forward**: walk left to right. at each visit, "how likely is everything so far if the patient is in each stage right now?"
- **backward**: walk right to left. "how likely is everything we'll see later, given each stage right now?"
- combine both: each visit's confidence uses evidence from the *entire* sequence

everything stays in log space so numbers don't underflow.

In [148]:
def _forward_backward(self, log_emission_probs):
    """for one patient's visits, figure out how confident we are about which disease stage
    each visit belongs to, using evidence from the entire sequence (not just past visits)."""
    seq_len, n_states = log_emission_probs.shape
    log_start = np.log(self.startprob_ + 1e-300)
    log_trans = np.log(self.transmat_ + 1e-300)

    # forward pass: walk left to right through visits
    log_forward = np.empty((seq_len, n_states))
    # first visit: just start probs times how well features match each stage
    log_forward[0] = log_start + log_emission_probs[0]
    for t in range(1, seq_len):
        # for each stage at visit t: sum over all possible previous stages
        # "could have come from any stage at t-1, transitioned here, and produced these features"
        log_forward[t] = log_emission_probs[t] + log_sum_exp(log_forward[t - 1, :, None] + log_trans, axis=0)

    # backward pass: walk right to left
    log_backward = np.empty((seq_len, n_states))
    # last visit: no future to account for
    log_backward[-1] = 0.0
    for t in range(seq_len - 2, -1, -1):
        # for each stage at visit t: sum over all possible next stages
        log_backward[t] = log_sum_exp(log_trans + log_emission_probs[t + 1, None, :] + log_backward[t + 1, None, :], axis=1)

    # total sequence likelihood (used to normalize into proper probabilities)
    log_total = log_sum_exp(log_forward[-1])
    # combine forward + backward to get per-visit stage confidences
    log_state_probs = log_forward + log_backward - log_total

    # joint probabilities for consecutive visit pairs (needed for updating transitions)
    if seq_len > 1:
        log_pair_probs = np.empty((seq_len - 1, n_states, n_states))
        for t in range(seq_len - 1):
            log_pair_probs[t] = (log_forward[t, :, None] + log_trans + log_emission_probs[t + 1, None, :] + log_backward[t + 1, None, :] - log_total)
    else:
        log_pair_probs = np.zeros((0, n_states, n_states))
    return log_state_probs, log_pair_probs, log_total

GaussianHMMScratch._forward_backward = _forward_backward

### training loop (Baum-Welch)

the main loop. goes through all patients, figures out which stage each visit probably belongs to, updates the model, repeats.

each round:
1. run forward-backward on every patient to get soft assignments (fractional counts based on confidence)
2. use those to recompute start probs, transitions, and emission distributions
3. check if total log likelihood improved. if barely, stop early

In [149]:
def fit(self, X, lengths):
    """the main training loop. goes through all patients, figures out which disease stage
    each visit probably belongs to, then updates the model's understanding of what each
    stage looks like. repeats until it stops improving."""
    n_states = self.n_states
    n_features = X.shape[1]
    prev_total = -np.inf

    for iteration in range(self.n_iter):
        # accumulators for the updated parameters
        sum_start_probs = np.zeros(n_states)
        sum_pair_probs  = np.zeros((n_states, n_states))
        sum_state_probs = np.zeros(n_states)
        sum_state_x     = np.zeros((n_states, n_features))
        sum_state_xx    = np.zeros((n_states, n_features))
        total_log_lik   = 0.0

        # run forward-backward on every patient and collect soft assignments
        offset = 0
        for length in lengths:
            patient_seq = X[offset:offset + length]
            offset += length
            log_emit = self._log_emission_probs(patient_seq)
            log_state_probs, log_pair_probs, log_lik = self._forward_backward(log_emit)
            # convert from log space to regular probabilities so we can add them up
            state_probs = np.exp(log_state_probs)
            pair_probs  = np.exp(log_pair_probs)
            # only the first visit contributes to start distribution
            sum_start_probs += state_probs[0]
            if pair_probs.shape[0] > 0:
                sum_pair_probs += pair_probs.sum(axis=0)
            sum_state_probs += state_probs.sum(axis=0)
            # weighted sums for computing new means and variances
            sum_state_x  += state_probs.T @ patient_seq
            sum_state_xx += state_probs.T @ (patient_seq * patient_seq)
            total_log_lik += log_lik

        # update start distribution: how often does each stage appear at visit 0?
        self.startprob_ = sum_start_probs / max(sum_start_probs.sum(), 1e-300)
        # update transition table: from stage i, where do patients go next on average?
        row_totals = sum_pair_probs.sum(axis=1, keepdims=True)
        row_totals = np.where(row_totals < 1e-300, 1.0, row_totals)
        self.transmat_ = sum_pair_probs / row_totals
        # update means: weighted average of features for each stage
        self.means_ = sum_state_x / np.maximum(sum_state_probs[:, None], 1e-300)
        # update variances: weighted average of squared deviations
        # floor variances so they never hit zero (would make the gaussian explode)
        self.covars_ = np.clip(sum_state_xx / np.maximum(sum_state_probs[:, None], 1e-300) - self.means_ ** 2,
            1e-4, None)

        # did we improve? if not (or barely), stop early
        if total_log_lik - prev_total < self.tol:
            break
        prev_total = total_log_lik

GaussianHMMScratch.fit = fit

### scoring

how well does the model fit the data? computes total log-likelihood across all patients. higher = less surprised = better fit.

used to compare random restarts and pick the winner. only needs the forward pass.

In [150]:
def score(self, X, lengths):
    """how well does the current model explain the data? higher = better fit.
    just runs the forward pass on every patient and sums up the log-likelihoods."""
    log_start = np.log(self.startprob_ + 1e-300)
    log_trans = np.log(self.transmat_  + 1e-300)
    total = 0.0
    offset = 0
    for length in lengths:
        patient_seq = X[offset:offset + length]
        offset += length
        log_emit = self._log_emission_probs(patient_seq)
        seq_len, n_states = log_emit.shape
        log_forward = np.empty((seq_len, n_states))
        log_forward[0] = log_start + log_emit[0]
        for t in range(1, seq_len):
            log_forward[t] = log_emit[t] + log_sum_exp(log_forward[t - 1, :, None] + log_trans, axis=0)
        # sum at the last visit = log probability of this patient's whole sequence
        total += float(log_sum_exp(log_forward[-1]))
    return total

GaussianHMMScratch.score = score

### Viterbi decoding

finds the single most likely sequence of stages for one patient (not a distribution, just the best guess).

forward sweep: at each visit, store the best score and which previous stage led to it. then backtrack from the best final stage to get the full path. like finding the shortest path through a graph.

In [151]:
def predict(self, X):
    """Viterbi decoding: find the single most likely sequence of disease stages for one patient."""
    log_emit = self._log_emission_probs(X)
    seq_len, n_states = log_emit.shape
    log_start = np.log(self.startprob_ +1e-300)
    log_trans = np.log(self.transmat_ +1e-300)

    viterbi_log = np.empty((seq_len, n_states))
    backpointer = np.zeros((seq_len, n_states), dtype=int)
    # first visit: just start probs times emission
    viterbi_log[0] = log_start + log_emit[0]

    # forward sweep: for each visit, find the best previous stage for each current stage
    for t in range(1, seq_len):
        scores = viterbi_log[t - 1, :, None] + log_trans
        backpointer[t] = np.argmax(scores, axis=0)
        viterbi_log[t] = log_emit[t] + scores[backpointer[t], np.arange(n_states)]

    # backtrack from the best final stage to recover the full path
    states = np.empty(seq_len, dtype=int)
    states[-1] = int(np.argmax(viterbi_log[-1]))
    for t in range(seq_len - 2, -1, -1):
        states[t] = backpointer[t + 1, states[t + 1]]
    return states

GaussianHMMScratch.predict = predict

## Naive Bayes baseline

a classifier with no concept of sequence. looks at each visit independently, ignoring order.

for each CDR class, fit a bell curve per feature. to predict: which class's bell curves fit this visit best?

"naive" = pretends features are independent (same assumption as HMM emissions, just without the temporal part). this is our main comparison: if the HMM can't beat NB, then the temporal modeling isn't adding value.

In [152]:
class GaussianNBScratch:
    """Gaussian Naive Bayes from scratch. Used as a static (non-sequence) baseline."""

    def fit(self, X, y):
        """Fit a per-class mean and variance for every feature."""
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        self.means_ = np.zeros((n_classes, n_features))
        self.variances_ = np.zeros((n_classes, n_features))
        self.log_priors_ = np.zeros(n_classes)
        for i, cls in enumerate(self.classes_):
            mask = (y == cls)
            self.means_[i] = X[mask].mean(axis=0)
            # floor variance for numerical safety
            self.variances_[i] = np.clip(X[mask].var(axis=0), 1e-6, None)
            # log P(class) = log fraction of visits in this class
            self.log_priors_[i] = np.log(mask.mean() + 1e-12)
        return self

    def predict(self, X):
        """Pick the class with the highest log P(class) + log P(visit | class)."""
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        log_two_pi = np.log(2.0 * np.pi)
        scores = np.empty((len(X), n_classes))
        for i in range(n_classes):
            diff = X - self.means_[i]
            variance = self.variances_[i]
            # log prior plus log of the gaussian pdf
            scores[:, i] = (self.log_priors_[i] - 0.5 * (n_features * log_two_pi + np.log(variance).sum() + ((diff * diff) / variance).sum(axis=1)))
        return self.classes_[np.argmax(scores, axis=1)]

## pipeline helpers
1. HMM training: warm start, single restart, multi-restart loop
2. decoding + alignment: Viterbi over sequences, Hungarian state-to-CDR map
3. data wrangling: pack patient sequences, count transitions, smooth transition matrix
4. the orchestrator: `run_hmm_pipeline` ties it all together

### part 1: HMM training (warm start + restarts)

`init_transition_from_labels` builds an initial transition matrix from observed CDR transitions. gives the model a reasonable starting point instead of random guesses.

the warm start from per-class means is legit:
- we use labels to *initialize* where clusters start, but Baum-Welch can move them wherever the data says
- without it, the model often converges to a bad solution where two states collapse onto the same CDR level

flags control the behavior:
- `use_warm_init`: if True, half the restarts start from per-CDR averages. if False, all restarts are random
- `use_dirichlet`: if True, initialize transitions from observed CDR transitions. if False, start with uniform transitions

In [153]:
def init_transition_from_labels(y, lengths, dirichlet=1.0):
    """build initial transition matrix from observed CDR transitions in training.
    counts how often CDR X is followed by CDR Y, adds dirichlet smoothing,
    normalizes each row to sum to 1."""
    class_idx = cdr_to_class_idx(y)
    counts = np.full((3, 3), dirichlet, dtype=float)
    offset = 0
    for length in lengths:
        seq = class_idx[offset:offset + length]
        offset += length
        for a, b in zip(seq[:-1], seq[1:]):
            counts[a, b] += 1
    return counts / counts.sum(axis=1, keepdims=True)


def fit_hmm_one_restart(X, y, lengths, seed, init_mode="warm", use_dirichlet=True):
    """one Baum-Welch run from a single starting point.
    warm mode: start each stage's mean at the actual average of that CDR class's visits.
    random mode: scatter the starting means randomly around the global average.
    use_dirichlet: if True, init transitions from observed CDR transitions. if False, uniform."""
    rng = np.random.default_rng(seed)
    class_idx = cdr_to_class_idx(y)
    n_states, n_features = 3, X.shape[1]
    global_mean = X.mean(axis=0)
    global_var = np.clip(X.var(axis=0), 1e-3, None)

    if init_mode == "warm":  # start means from per-CDR class averages (gives the model a hint)
        # start each state at the average of its CDR class
        means = np.zeros((n_states, n_features))
        variances = np.zeros((n_states, n_features))
        for state in range(n_states):
            in_class = class_idx == state
            if in_class.sum() < 2:
                # rare class: fall back to global stats
                means[state]     = global_mean
                variances[state] = global_var
            else:
                means[state]     = X[in_class].mean(axis=0)
                variances[state] = np.clip(X[in_class].var(axis=0), 1e-3, None)
        # small jitter so warm restarts aren't all identical
        means += rng.normal(0, 0.05, means.shape)
    else:
        # random: scatter 3 state means around the global mean
        means = global_mean + rng.normal(0, 0.6, (n_states, n_features))
        variances = np.tile(global_var, (n_states, 1))

    hmm_model = GaussianHMMScratch(n_states=n_states, n_iter=200, tol=1e-4, random_state=int(rng.integers(1 << 30)))
    hmm_model.startprob_ = np.full(n_states, 1.0 / n_states)

    # transition matrix: either from observed CDR transitions or uniform
    if use_dirichlet:
        hmm_model.transmat_ = init_transition_from_labels(y, lengths)
    else:
        hmm_model.transmat_ = np.full((n_states, n_states), 1.0 / n_states)

    hmm_model.means_  = means
    hmm_model.covars_ = variances
    hmm_model.fit(X, lengths)
    return hmm_model


# try multiple random starting points and keep the best model
# (EM can get stuck in bad local optima, restarts help avoid that)
def fit_hmm_with_restarts(X, y, lengths, n_restarts=8, base_seed=42,
                          use_warm_init=True, use_dirichlet=True):
    """run Baum-Welch from multiple starting points and keep the best.
    picks the winner by training agreement (how many visits it classified correctly
    after Hungarian alignment), not raw log-likelihood. prevents restarts that
    fit well in an unsupervised sense but collapse rare classes.

    use_warm_init: if True, half restarts use per-CDR averages, half random.
                   if False, all restarts are random.
    use_dirichlet: if True, init transitions from observed CDR transitions.
                   if False, start with uniform transitions.
    n_restarts: how many random restarts to try."""
    best_score = -np.inf
    best_model = None
    best_log_lik = -np.inf
    for restart_idx in range(n_restarts):
        # decide warm vs random for this restart
        if use_warm_init:
            mode = "warm" if restart_idx < n_restarts // 2 else "random"
        else:
            mode = "random"
        candidate_model = fit_hmm_one_restart(
            X, y, lengths, seed=base_seed + 997 * restart_idx,
            init_mode=mode, use_dirichlet=use_dirichlet)
        # score by training agreement after hungarian matching
        train_states = predict_state_sequence(candidate_model, X, lengths)
        _, train_agreement = hungarian_state_to_cdr_map(train_states, y)
        if train_agreement > best_score:
            best_score = train_agreement
            best_model = candidate_model
            best_log_lik = candidate_model.score(X, lengths)
    return best_model, best_log_lik

### part 2: decoding and alignment
- `predict_state_sequence`: run Viterbi on every patient and concatenate results
- `hungarian_state_to_cdr_map`: finds best one-to-one mapping from hidden states to CDR levels. builds a confusion table, then picks the assignment that maximizes agreement. standard approach for evaluating unsupervised models

In [154]:
def predict_state_sequence(hmm_model, X, lengths):
    """run Viterbi on every patient and combine the predicted stage sequences."""
    all_states = []
    offset = 0
    for length in lengths:
        all_states.append(hmm_model.predict(X[offset:offset + length]))
        offset += length
    return np.concatenate(all_states)

def hungarian_state_to_cdr_map(predicted_states, true_cdr):
    """find the best one-to-one mapping from hidden states to CDR levels.
    builds a table of how often each state co-occurred with each CDR class,
    then uses the Hungarian algorithm to pick the assignment that maximizes agreement."""
    match_counts = np.zeros((3, 3))
    for state, true_class in zip(predicted_states, cdr_to_class_idx(true_cdr)):
        match_counts[state, true_class] += 1
    # scipy minimizes cost, so negate to maximize agreement instead
    rows, cols = linear_sum_assignment(-match_counts)  # hungarian: best 1-to-1 state-to-CDR mapping
    state_to_cdr = {int(rows[i]): float(CDR_VALUES[cols[i]]) for i in range(3)}
    train_agreement = float(match_counts[rows, cols].sum()) / max(len(true_cdr), 1)
    return state_to_cdr, train_agreement

### part 3: data helpers

- `pack_patient_sequences`: groups by patient, scales features, stacks into one big array + lengths list
- `count_observed_cdr_transitions`: counts CDR X followed by CDR Y in training
- `smooth_transition_matrix`: adds Dirichlet fake counts before normalizing. prevents any transition from being exactly zero. fake counts encode our belief that dementia usually stays the same or gets worse

In [155]:
def pack_patient_sequences(sequence_df, feature_cols, scaler, patient_ids):
    """group visits by patient, scale features, and stack into one big array.
    also returns a lengths list so we know where each patient's chunk starts and ends."""
    feature_chunks, label_chunks, lengths = [], [], []
    for patient_id, group in sequence_df.groupby("Subject ID"):
        # dropping patients with <2 visits (cant learn transitions from a single visit)
        if len(group) < 2 or patient_id not in patient_ids:
            continue
        feature_chunks.append(scaler.transform(group[feature_cols].to_numpy(float)))
        label_chunks.append(group["CDR"].to_numpy(float))
        lengths.append(len(group))
    if not feature_chunks:
        return np.empty((0, len(feature_cols))), np.array([]), np.array([], dtype=int)
    return np.vstack(feature_chunks), np.concatenate(label_chunks), np.array(lengths, dtype=int)


def count_observed_cdr_transitions(sequence_df, patient_ids):
    """count CDR-to-CDR transitions between consecutive visits in training."""
    counts = np.zeros((3, 3))
    for patient_id, group in sequence_df.groupby("Subject ID"):
        if patient_id not in patient_ids or len(group) < 2:
            continue
        idx_seq = cdr_to_class_idx(group.sort_values("Visit")["CDR"].to_numpy())
        for current_idx, next_idx in zip(idx_seq[:-1], idx_seq[1:]):
            counts[current_idx, next_idx] += 1
    return counts

def smooth_transition_matrix(observed_counts, dirichlet_prior):
    """add dirichlet fake counts to observed transitions and renormalize.
    prevents any transition probability from being exactly zero."""
    posterior_counts = observed_counts + dirichlet_prior
    return posterior_counts / posterior_counts.sum(axis=1, keepdims=True)

### part 4: the full pipeline

`run_hmm_pipeline` ties everything together: scale features, optionally PCA, train HMM with restarts, decode test patients, compare against baselines.

flags:
- `use_pca`: reduce dimensions and decorrelate features. essential for high-dim CNN features, optional for clinical
- `use_warm_init`: start from per-CDR averages (True) or fully random (False)
- `use_dirichlet`: init transitions from observed CDR transitions (True) or uniform (False)
- `n_restarts`: how many random restarts to try

In [156]:
def run_hmm_pipeline(name, sequence_df, feature_cols, train_patient_ids, test_patient_ids,
                     n_pca_components=8, use_pca=False, use_warm_init=True,
                     use_dirichlet=True, n_restarts=8, show=True):
    """end-to-end HMM pipeline on one feature set.
    handles scaling, optional PCA, training with restarts, decoding, baselines.

    flags:
      use_pca: PCA whitening before HMM (good for high-dim features)
      use_warm_init: start from per-CDR averages (True) or all random (False)
      use_dirichlet: init transitions from observed CDR transitions (True) or uniform (False)
      n_restarts: how many random restarts"""
    # standardize on training rows only so test stats don't leak in
    train_rows = sequence_df["Subject ID"].isin(train_patient_ids)
    # scale using training stats only (no data leakage from test)
    feature_scaler = StandardScaler().fit(sequence_df.loc[train_rows, feature_cols].to_numpy(float))
    X_train, y_train, train_lengths = pack_patient_sequences(sequence_df, feature_cols, feature_scaler, train_patient_ids)
    X_test,  y_test,  test_lengths  = pack_patient_sequences(sequence_df, feature_cols, feature_scaler, test_patient_ids)

    # optional PCA: reduces dimensions and decorrelates features
    if use_pca:
        n_components = min(n_pca_components, X_train.shape[1])
        pca = PCA(n_components=n_components, whiten=True, random_state=42)
        X_train_pca = pca.fit_transform(X_train)
        X_test_pca  = pca.transform(X_test)
    else:
        X_train_pca = X_train
        X_test_pca  = X_test

    # train the HMM with random restarts, keep the best
    best_model, best_log_lik = fit_hmm_with_restarts(
        X_train_pca, y_train, train_lengths,
        n_restarts=n_restarts, use_warm_init=use_warm_init, use_dirichlet=use_dirichlet)

    # figure out which hidden state maps to which CDR level
    train_state_seq = predict_state_sequence(best_model, X_train_pca, train_lengths)
    state_to_cdr, train_agreement = hungarian_state_to_cdr_map(train_state_seq, y_train)

    # decode test set with Viterbi and translate states to CDR
    test_state_seq = predict_state_sequence(best_model, X_test_pca, test_lengths)
    y_test_pred = np.array([state_to_cdr[state] for state in test_state_seq], float)
    y_test_idx = cdr_to_class_idx(y_test)
    y_pred_idx = cdr_to_class_idx(y_test_pred)

    confusion = confusion_matrix(y_test_idx, y_pred_idx, labels=[0, 1, 2])
    hmm_macro_f1 = float(f1_score(y_test_idx, y_pred_idx, labels=[0, 1, 2], average="macro", zero_division=0.0))
    hmm_balanced = float(balanced_accuracy_score(y_test_idx, y_pred_idx))

    # majority baseline: predict the most common class for everything
    most_common_class = int(np.bincount(cdr_to_class_idx(y_train), minlength=3).argmax())
    y_pred_majority = np.full_like(y_test_idx, most_common_class)
    majority_macro_f1 = float(f1_score(y_test_idx, y_pred_majority, labels=[0, 1, 2], average="macro", zero_division=0.0))
    majority_balanced = float(balanced_accuracy_score(y_test_idx, y_pred_majority))

    # naive bayes baseline: same features, no sequence modeling
    naive_bayes_model = GaussianNBScratch().fit(X_train_pca, cdr_to_class_idx(y_train))
    y_pred_nb = naive_bayes_model.predict(X_test_pca)
    nb_macro_f1 = float(f1_score(y_test_idx, y_pred_nb, labels=[0, 1, 2], average="macro", zero_division=0.0))
    nb_balanced = float(balanced_accuracy_score(y_test_idx, y_pred_nb))

    if show:
        print(f"{name}")
        print(f"features: {len(feature_cols)} columns | train/test visits: {len(y_train)} / {len(y_test)}")
        print(f"best training log-likelihood: {best_log_lik:.2f}")
        print(f"state -> cdr (hungarian): {state_to_cdr}  (train agreement {train_agreement:.1%})")
        print()
        print(classification_report(y_test_idx, y_pred_idx, labels=[0, 1, 2], target_names=["CDR 0", "CDR 0.5", "CDR 1"], digits=3, zero_division=0.0))
        print(f"  {'majority class':22}  macro f1 = {majority_macro_f1:.3f}   balanced acc = {majority_balanced:.3f}")
        print(f"  {'gaussian naive bayes':22}  macro f1 = {nb_macro_f1:.3f}   balanced acc = {nb_balanced:.3f}")
        print(f"  {'hmm + viterbi':22}  macro f1 = {hmm_macro_f1:.3f}   balanced acc = {hmm_balanced:.3f}")

        fig, ax = plt.subplots(figsize=(5.2, 4.2))
        sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", xticklabels=["CDR 0", "CDR 0.5", "CDR 1"], yticklabels=["CDR 0", "CDR 0.5", "CDR 1"], cbar=False, ax=ax, annot_kws={"fontsize": 12})
        ax.set_xlabel("predicted cdr")
        ax.set_ylabel("true cdr")
        ax.set_title(f"{name}: HMM confusion matrix (held-out patients)")
        plt.tight_layout()
        plt.show()

    return {
        "name": name,
        "n_features": len(feature_cols),
        "n_visits_train": int(len(y_train)),
        "n_visits_test": int(len(y_test)),
        "train_log_likelihood": float(best_log_lik),
        "state_to_cdr": state_to_cdr,
        "train_agreement": train_agreement,
        "confusion_matrix": confusion,
        "hmm_macro_f1": hmm_macro_f1,
        "hmm_balanced_accuracy": hmm_balanced,
        "majority_macro_f1": majority_macro_f1,
        "majority_balanced_accuracy": majority_balanced,
        "gnb_macro_f1": nb_macro_f1,
        "gnb_balanced_accuracy": nb_balanced,
        "model": best_model,
    }